In [5]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Configuration
// - BM, BN: size of the tile, work for each Thread Block
// - BK: step size, how many elements are loaded together for the multiplication
// - TM, TN: size of the block computed by each single thread
#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// ---- KERNEL CUDA: 2D Register Tiling ----
__global__ void CudaKernel(int n, const float *A, const float *B, float *C) {
    // Memoria Condivisa per i Tile
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Starting index for the block 8x8 of this thread
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    // Registers to store the partial results of the multiplication
    float res[TM][TN] = {0.0f};

    // Linear index of the thread for Decoupled Loading
    int tid = ty * (BN / TN) + tx;

    // Loop
    for (int p = 0; p < (n + BK - 1) / BK; ++p) {
        // Decoupled Loading: 256 threads loads the data from the Global to the Shared

        // Loading of the Tile A
        for (int i = 0; i < (BM * BK) / 256; ++i) {
            int idx = i * 256 + tid;
            int a_row = idx / BK;
            int a_col = idx % BK;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col;
            sA[a_row][a_col] = (g_a_row < n && g_a_col < n) ? A[g_a_row * n + g_a_col] : 0.0f;
        }

        // Loading of the Tile B
        for (int i = 0; i < (BK * BN) / 256; ++i) {
            int idx = i * 256 + tid;
            int b_row = idx / BN;
            int b_col = idx % BN;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col;
            sB[b_row][b_col] = (g_b_row < n && g_b_col < n) ? B[g_b_row * n + g_b_col] : 0.0f;
        }

        // Synchronization Barrier: ensures that all threads have finished loading the data before to start the computation
        __syncthreads();

        // Computation in the Registers
        #pragma unroll
        for (int k = 0; k < BK; ++k) {

            float a_reg[TM];
            float b_reg[TN];

            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    // Saving in the Global Memory
    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {
    if (argc < 2) return 1;
    int n = atoi(argv[1]);
    if (n <= 0) return 1;

    // CPU Memory Allocation - Flattening 1D: ensures that the memory is contiguous
    size_t bytes = n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // GPU Memory Allocation & Transfering
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy from CPU to GPU: bottleneck
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Definition of Grid and Block for the execution - Ceiling approach
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    // Variables for measuring the execution time
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Time Measurement
    cudaEventRecord(start);
    CudaKernel<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);
    cudaEventRecord(stop);

    // Synchronization: CPU waits for the GPU to finish
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Fast Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy from GPU to CPU (RAM)
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res-fast.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation of both CPU and GPU
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('my_cuda.cu', 'w') as f:
    f.write(cell_str)

In [6]:
!nvcc -arch=sm_75 -O3 my_cuda.cu -o my_cuda

In [8]:
!nvprof ./my_cuda 20000

==1245== NVPROF is profiling process 1245, command: ./my_cuda 20000
Fast Execution Time: 4.0186 seconds
==1245== Profiling application: ./my_cuda 20000
==1245== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   79.51%  4.01837s         1  4.01837s  4.01837s  4.01837s  CudaKernel(int, float const *, float const *, float*)
                   13.35%  674.96ms         2  337.48ms  336.38ms  338.59ms  [CUDA memcpy HtoD]
                    7.14%  360.72ms         1  360.72ms  360.72ms  360.72ms  [CUDA memcpy DtoH]
      API calls:   76.68%  4.01838s         1  4.01838s  4.01838s  4.01838s  cudaEventSynchronize
                   19.78%  1.03659s         3  345.53ms  336.60ms  361.22ms  cudaMemcpy
                    3.37%  176.81ms         3  58.938ms  164.29us  176.48ms  cudaMalloc
                    0.11%  5.9442ms         3  1.9814ms  1.0776ms  2.7922ms  cudaFree
                    0.05%  2.4674ms       114  21.643us 